# 1. Collect rollouts and judge them

Runs the base Cosmos Policy on the chosen tasks with several rollout seeds per start state, scores every rollout with both Robometer judges, and builds the fine-tune set (the judge's best attempt per start state).

**Before running**
- Runtime: A100 GPU.
- Add a Colab secret named `HF_TOKEN` from a Hugging Face account that has accepted the license for `nvidia/Cosmos-Predict2-2B-Video2World` (the policy needs its tokenizer).

The notebook is self-contained: the code cells below write the scripts to `/content/code`, and the run cells call them. The scripts run as subprocesses because the policy and the judges each need their own Python environment (their torch/transformers versions conflict).

**Time on an A100:** setup ~20 min; building the tasks ~15 min; ~45 min per seed on the 5 custom tasks (125 episodes); judging ~3 s per rollout per judge.

Everything that must survive the session (the built tasks, episode logs, judge scores, the fine-tune set) is written to `OUT` on Drive. Full trajectories stay on the VM.

In [ ]:
OUT = '/content/drive/MyDrive/cosmos_robometer/runs'
TASKS = 'custom'
SEEDS = 8
HELDOUT_SEEDS = 2
SELECT_WITH = 'libero'

`TASKS`: `custom` = the 5 cluttered tasks, or a LIBERO suite name such as `libero_spatial`. `SEEDS`: attempts per start state (best-of-N). `SELECT_WITH`: which judge picks the fine-tune set (`libero` or `base`).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
CODE = '/content/code'
os.makedirs(f'{CODE}/tasks', exist_ok=True)
TASK_ARGS = f'--tasks_manifest {CODE}/tasks/manifest.json' if TASKS == 'custom' else f'--suite {TASKS}'
os.environ.update(CODE=CODE, OUT=OUT, TASK_ARGS=TASK_ARGS)
os.makedirs(OUT, exist_ok=True)
os.environ.update(SEEDS=str(SEEDS), HELDOUT_SEEDS=str(HELDOUT_SEEDS), SELECT_WITH=SELECT_WITH)

## Code

VM setup: repos, environments, weights, the gated tokenizer, and the LIBERO and EGL config.

In [ ]:
%%writefile /content/code/setup_colab.sh
#!/bin/bash
set -e
WANT_JUDGE=${1:-0}
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update
apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libglib2.0-0 libosmesa6 ffmpeg > /dev/null
pip -q install uv
cd /content
[ -d cosmos-policy ] || git clone -q --depth 1 https://github.com/NVlabs/cosmos-policy.git
[ -d LIBERO ] || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
(cd cosmos-policy && uv sync -q --extra cu128 --group libero --python 3.10) &
PIDS=$!
if [ "$WANT_JUDGE" = 1 ]; then
  [ -d robometer ] || git clone -q --depth 1 https://github.com/robometer/robometer.git
  (cd robometer && uv sync -q && uv pip install -q --python .venv/bin/python "torchao==0.13.0" h5py pyyaml pillow) &
  PIDS="$PIDS $!"
fi
hf download nvidia/Cosmos-Policy-LIBERO-Predict2-2B > /dev/null
hf download nvidia/Cosmos-Predict2-2B-Video2World tokenizer/tokenizer.pth > /dev/null
for p in $PIDS; do wait $p; done

cat > /content/env.sh <<'EOF'
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
export __EGL_VENDOR_LIBRARY_FILENAMES=/content/10_nvidia.json
export LD_LIBRARY_PATH=/usr/lib64-nvidia:$LD_LIBRARY_PATH
export CPV=/content/cosmos-policy/.venv/bin/python
export RBV=/content/robometer/.venv/bin/python
EOF
echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "/usr/lib64-nvidia/libEGL_nvidia.so.0"}}' > /content/10_nvidia.json
CPV=/content/cosmos-policy/.venv/bin/python
L=$($CPV -c "import importlib.util, os; print(os.path.dirname(importlib.util.find_spec('libero').origin))")
[ -d "$L/libero" ] && L=$L/libero
mkdir -p /root/.libero
printf "benchmark_root: %s\nbddl_files: %s/bddl_files\ninit_states: %s/init_files\ndatasets: %s/../datasets\nassets: /content/LIBERO/libero/libero/assets\n" "$L" "$L" "$L" "$L" > /root/.libero/config.yaml
[ -e $L/assets ] || ln -s /content/LIBERO/libero/libero/assets $L/assets
echo SETUP_DONE

The 5 cluttered tasks: same instructions as LIBERO, lookalike distractors added. `make_tasks.py` builds the task files and samples 50 start states per task, rejecting any where objects fall, slide, wobble, tip over or interpenetrate, the task is already done, or the target is off camera. It is deterministic: it reproduces the exact start states used in our results.

In [ ]:
%%writefile /content/code/tasks/spec.json
[
{"name": "bowl_plate", "source": "libero_goal/put_the_bowl_on_the_plate.bddl", "language": "put the bowl on the plate", "table": "main_table", "clutter": [["butter", [0.15, 0.1]], ["cookies", [0.2, -0.02]], ["chocolate_pudding", [0.0, 0.24]], ["ketchup", [0.13, -0.08]], ["chocolate_pudding", [-0.28, 0.02]]], "must_see": ["akita_black_bowl_1", "plate_1"]},
{"name": "cheese_bowl", "source": "libero_goal/put_the_cream_cheese_in_the_bowl.bddl", "language": "put the cream cheese in the bowl", "table": "main_table", "clutter": [["butter", [0.15, 0.1]], ["butter", [0.13, -0.08]], ["chocolate_pudding", [0.0, 0.25]], ["ketchup", [0.18, 0.22]]], "must_see": ["cream_cheese_1", "akita_black_bowl_1"]},
{"name": "bowl_stove", "source": "libero_goal/put_the_bowl_on_the_stove.bddl", "language": "put the bowl on the stove", "table": "main_table", "clutter": [["tomato_sauce", [0.0, 0.25]], ["cookies", [0.2, -0.02]], ["ketchup", [0.15, 0.12]], ["chocolate_pudding", [0.13, -0.08]]], "must_see": ["akita_black_bowl_1", "flat_stove_1"]},
{"name": "bottle_cabinet", "source": "libero_goal/put_the_wine_bottle_on_top_of_the_cabinet.bddl", "language": "put the wine bottle on top of the cabinet", "table": "main_table", "clutter": [["ketchup", [0.15, 0.1]], ["salad_dressing", [0.0, 0.25]], ["bbq_sauce", [-0.2, 0.08]], ["ketchup", [0.13, -0.08]]], "must_see": ["wine_bottle_1", "wooden_cabinet_1"]},
{"name": "ketchup_basket", "source": "libero_object/pick_up_the_ketchup_and_place_it_in_the_basket.bddl", "language": "pick up the ketchup and place it in the basket", "table": "floor", "clutter": [["bbq_sauce", [-0.01, -0.25]], ["bbq_sauce", [0.22, -0.12]], ["salad_dressing", [-0.05, 0.1]], ["tomato_sauce", [-0.25, 0.2]], ["wooden_tray", [0.22, 0.14]]], "must_see": ["ketchup_1", "basket_1"]}
]

In [ ]:
%%writefile /content/code/tasks/make_tasks.py
import json
import os
import pickle
import re
import sys

os.environ.setdefault('MUJOCO_GL', 'egl')
import numpy as np
from PIL import Image
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from robosuite.utils.camera_utils import get_camera_transform_matrix, project_points_from_world_to_camera

HERE = os.path.dirname(os.path.abspath(__file__))
N_STATES = int(sys.argv[1]) if len(sys.argv) > 1 else 50
SPECS = json.load(open(os.path.join(HERE, 'spec.json')))
JITTER = 0.015


def build_bddl(spec):
    s = open(os.path.join(get_libero_path('bddl_files'), spec['source'])).read()
    s = re.sub(r'\(:language [^\n]*\)', f"(:language {spec['language']})", s)
    table = spec['table']
    regions, objects, inits, count = [], [], [], {}
    for i, (cls, (x, y)) in enumerate(spec['clutter']):
        count[cls] = count.get(cls, 1) + 1
        name, region = f'{cls}_{count[cls]}', f'clutter_region_{i}'
        box = ' '.join(str(round(v, 4)) for v in (x - JITTER, y - JITTER, x + JITTER, y + JITTER))
        regions.append(f'      ({region}\n          (:target {table})\n          (:ranges (\n              ({box})\n            )\n          )\n      )')
        objects.append(f'    {name} - {cls}')
        inits.append(f'    (On {name} {table}_{region})')
    s = s.replace('(:regions\n', '(:regions\n' + '\n'.join(regions) + '\n', 1)
    block = re.search(r'\(:objects\n(.*?)\n  \)', s, re.S)
    by_class = {}
    for line in block.group(1).split('\n') + objects:
        names, cls = line.strip().split(' - ')
        by_class.setdefault(cls.strip(), []).extend(names.split())
    s = s[:block.start(1)] + '\n'.join(f"    {' '.join(v)} - {k}" for k, v in by_class.items()) + s[block.end(1):]
    return s.replace('(:init\n', '(:init\n' + '\n'.join(inits) + '\n', 1)


def check(env, spec):
    sim = env.env.sim
    m = sim.model
    bodies = [(m.jnt_bodyid[j], m.body_id2name(m.jnt_bodyid[j])) for j in range(m.njnt) if m.jnt_type[j] == 0]
    spawn = {b: (sim.data.body_xpos[b].copy(), sim.data.body_xmat[b].reshape(3, 3)[:, 2].copy()) for b, _ in bodies}
    if env.env._check_success():
        return 'success_at_start', None
    for _ in range(10):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
    image = obs['agentview_image'][::-1].copy()
    settled = {b: sim.data.body_xpos[b].copy() for b, _ in bodies}
    for _ in range(40):
        env.step([0, 0, 0, 0, 0, 0, -1])
    if env.env._check_success():
        return 'success_after_settle', None
    for b, name in bodies:
        z = sim.data.body_xmat[b].reshape(3, 3)[:, 2]
        if sim.data.body_xpos[b][2] < spawn[b][0][2] - 0.15:
            return f'fell_off:{name}', None
        if np.linalg.norm(settled[b][:2] - spawn[b][0][:2]) > 0.03:
            return f'slid:{name}', None
        if np.linalg.norm(sim.data.body_xpos[b] - settled[b]) > 0.01:
            return f'unstable:{name}', None
        if np.degrees(np.arccos(np.clip(np.dot(z, spawn[b][1]), -1, 1))) > 30:
            return f'tipped:{name}', None
    robot = lambda g: (m.body_id2name(m.body_rootid[m.geom_bodyid[g]]) or '').lower().startswith(('robot', 'mount', 'gripper'))
    for i in range(sim.data.ncon):
        c = sim.data.contact[i]
        if c.dist < -0.004 and not (robot(c.geom1) or robot(c.geom2)):
            return 'penetration', None
    T = get_camera_transform_matrix(sim, 'agentview', 256, 256)
    for name in spec['must_see']:
        b = [b for b in range(m.nbody) if (m.body_id2name(b) or '').startswith(name)][0]
        r, c = project_points_from_world_to_camera(sim.data.body_xpos[b].reshape(1, 3), T, 256, 256)[0]
        if not (8 <= r < 248 and 8 <= c < 248):
            return f'offscreen:{name}', None
    return 'ok', image


for d in ('bddl', 'init', 'preview'):
    os.makedirs(os.path.join(HERE, d), exist_ok=True)
manifest = []
for spec in SPECS:
    name = spec['name']
    bddl = os.path.join(HERE, 'bddl', f'{name}.bddl')
    open(bddl, 'w').write(build_bddl(spec))
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    states, images, rejected, tries = [], [], {}, 0
    while len(states) < N_STATES and tries < 4 * N_STATES:
        tries += 1
        env.reset()
        state = np.asarray(env.get_sim_state()).copy()
        env.reset()
        env.set_init_state(state)
        why, image = check(env, spec)
        if why != 'ok':
            rejected[why.split(':')[0]] = rejected.get(why.split(':')[0], 0) + 1
            continue
        states.append(state)
        if len(images) < 8:
            images.append(image)
    env.close()
    pickle.dump(states, open(os.path.join(HERE, 'init', f'{name}.pkl'), 'wb'))
    if images:
        Image.fromarray(np.concatenate(images, 1)).save(os.path.join(HERE, 'preview', f'{name}.png'))
    manifest.append(dict(name=name, language=spec['language'], bddl=f'bddl/{name}.bddl', init=f'init/{name}.pkl'))
    print(name, f'{len(states)} valid of {tries}', 'rejected', rejected, flush=True)
json.dump(manifest, open(os.path.join(HERE, 'manifest.json'), 'w'), indent=1)

Policy rollouts with the 10 cm scene shift. Writes `episodes.jsonl` and, with `--save_traj 1`, one `.hdf5` trajectory per episode.

In [ ]:
%%writefile /content/code/rollout.py
import argparse
import json
import os
import pickle
from types import SimpleNamespace

import h5py
import numpy as np

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
os.environ['DETERMINISTIC'] = 'True'

import cosmos_policy._src.imaginaire.utils.checkpoint_db as checkpoint_db

_download = checkpoint_db.get_checkpoint_by_hf
_UNUSED = ('Cosmos-Predict2-2B-Video2World/model-480p-16fps.pt', 'Cosmos-Policy-ALOHA-Predict2-2B/Cosmos-Policy-ALOHA-Predict2-2B.pt')
checkpoint_db.get_checkpoint_by_hf = lambda uri: uri if uri.endswith(_UNUSED) else _download(uri)

from libero.libero import benchmark
from cosmos_policy.experiments.robot.cosmos_utils import get_action, get_model, init_t5_text_embeddings_cache, load_dataset_stats
from cosmos_policy.experiments.robot.libero.libero_utils import get_libero_dummy_action, get_libero_env
from cosmos_policy.experiments.robot.libero.run_libero_eval import TASK_MAX_STEPS, PolicyEvalConfig, prepare_observation
from cosmos_policy.utils.utils import jpeg_encode_image, set_seed_everywhere

p = argparse.ArgumentParser()
p.add_argument('--out', required=True)
p.add_argument('--suite', default='libero_spatial')
p.add_argument('--tasks_manifest', default='', help='tasks/manifest.json to run the custom cluttered tasks instead of a suite')
p.add_argument('--tasks', default='all')
p.add_argument('--episodes', default='25:50')
p.add_argument('--rseed', type=int, default=0)
p.add_argument('--scene_shift', type=float, default=0.10)
p.add_argument('--ckpt', default='nvidia/Cosmos-Policy-LIBERO-Predict2-2B')
p.add_argument('--save_traj', type=int, default=0)
a = p.parse_args()

HF = 'nvidia/Cosmos-Policy-LIBERO-Predict2-2B'
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only', ckpt_path=a.ckpt, config_file='cosmos_policy/config/config.py',
    dataset_stats_path=f'{HF}/libero_dataset_statistics.json', t5_text_embeddings_path=f'{HF}/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True, use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5, num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=a.suite, seed=195, deterministic=True)
stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)

if a.tasks_manifest:
    root = os.path.dirname(os.path.abspath(a.tasks_manifest))
    custom = json.load(open(a.tasks_manifest))
    n_tasks, max_steps = len(custom), 300
else:
    suite = benchmark.get_benchmark_dict()[a.suite]()
    n_tasks, max_steps = suite.n_tasks, TASK_MAX_STEPS[a.suite]
task_ids = range(n_tasks) if a.tasks == 'all' else [int(x) for x in a.tasks.split(',')]
e0, e1 = [int(x) for x in a.episodes.split(':')]


def load_task(tid):
    if not a.tasks_manifest:
        return suite.get_task(tid), suite.get_task_init_states(tid)
    t = custom[tid]
    bddl = os.path.join(root, t['bddl'])
    task = SimpleNamespace(language=t['language'], problem_folder=os.path.dirname(bddl), bddl_file=os.path.basename(bddl))
    return task, pickle.load(open(os.path.join(root, t['init']), 'rb'))


def shift_scene(env, rng):
    sim = env.env.sim
    m = sim.model
    free = [(j, m.jnt_bodyid[j]) for j in range(m.njnt) if m.jnt_type[j] == 0]
    free_bodies = {b for _, b in free}
    robotish = lambda name: name.startswith(('robot', 'mount', 'gripper'))
    fixtures = [b for b in range(1, m.nbody) if m.body_parentid[b] == 0 and b not in free_bodies
                and not robotish((m.body_id2name(b) or '').lower())
                and 'table' not in (m.body_id2name(b) or '').lower() and 'floor' not in (m.body_id2name(b) or '').lower()]
    touches_robot = lambda g: robotish((m.body_id2name(m.body_rootid[m.geom_bodyid[g]]) or '').lower())

    def contacts():
        sim.forward()
        cs = [sim.data.contact[k] for k in range(sim.data.ncon)]
        deep = sum(c.dist < -0.004 for c in cs)
        robot = sum(c.dist < -0.001 and (touches_robot(c.geom1) or touches_robot(c.geom2)) for c in cs)
        return deep, robot

    base_deep, _ = contacts()
    body_pos, qpos = m.body_pos.copy(), sim.data.qpos.copy()
    for _ in range(50):
        th = rng.uniform(0, 2 * np.pi)
        d = a.scene_shift * np.array([np.cos(th), np.sin(th)])
        for b in fixtures:
            m.body_pos[b, :2] = body_pos[b, :2] + d
        for j, _ in free:
            q = m.jnt_qposadr[j]
            sim.data.qpos[q:q + 2] = qpos[q:q + 2] + d
        deep, robot = contacts()
        if robot == 0 and deep <= base_deep:
            return True
    m.body_pos[:] = body_pos
    sim.data.qpos[:] = qpos
    sim.forward()
    return False


os.makedirs(os.path.join(a.out, 'traj'), exist_ok=True)
results = []
for tid in task_ids:
    task, init_states = load_task(tid)
    env, desc = get_libero_env(task, 'cosmos', resolution=256)
    for ep in range(e0, e1):
        env.reset()
        obs = env.set_init_state(init_states[ep])
        shifted = shift_scene(env, np.random.default_rng([0, tid, ep])) if a.scene_shift > 0 else False
        for _ in range(10):
            obs, _, _, _ = env.step(get_libero_dummy_action('cosmos'))
        prim, wrist, prop, acts, queue, success, t = [], [], [], [], [], False, 10
        while t < max_steps + 10:
            o = prepare_observation(obs, None, True)
            if not queue:
                set_seed_everywhere(a.rseed)
                r = get_action(cfg, model, stats, o, desc, seed=195 + a.rseed, randomize_seed=False,
                               num_denoising_steps_action=cfg.num_denoising_steps_action,
                               generate_future_state_and_value_in_parallel=True, batch_size=1)
                queue = [np.asarray(x) for x in r['actions']]
            act = queue.pop(0)
            prim.append(o['primary_image'])
            wrist.append(o['wrist_image'])
            prop.append(o['proprio'])
            acts.append(act)
            obs, _, done, _ = env.step(act.tolist())
            if done:
                success = True
                break
            t += 1
        row = dict(task=tid, ep=ep, rseed=a.rseed, success=success, steps=t, shifted=shifted)
        results.append(row)
        print(json.dumps(row), flush=True)
        with open(os.path.join(a.out, 'episodes.jsonl'), 'a') as f:
            f.write(json.dumps(row) + '\n')
        if a.save_traj:
            with h5py.File(os.path.join(a.out, 'traj', f'task{tid}_ep{ep}_r{a.rseed}_success{int(success)}.hdf5'), 'w') as f:
                jpeg = h5py.vlen_dtype(np.dtype('uint8'))
                f.create_dataset('primary_images_jpeg', data=[jpeg_encode_image(x, quality=95) for x in prim], dtype=jpeg)
                f.create_dataset('wrist_images_jpeg', data=[jpeg_encode_image(x, quality=95) for x in wrist], dtype=jpeg)
                f.create_dataset('proprio', data=np.stack(prop))
                f.create_dataset('actions', data=np.stack(acts))
                f.attrs['success'] = success
                f.attrs['task_description'] = desc
    env.close()
print(f'SUCCESS {np.mean([r["success"] for r in results]):.4f} over {len(results)} episodes', flush=True)

Robometer judge (`base` or `libero`): 8 frames of the main camera, giving per-frame progress and success probability.

In [ ]:
%%writefile /content/code/judge.py
import argparse
import glob
import io
import json
import os
import re
import sys
import types
from dataclasses import fields
from pathlib import Path

import h5py
import numpy as np
import torch
import yaml
from PIL import Image

JUDGES = {'base': 'robometer/Robometer-4B', 'libero': 'jesbu1/robometer-4b-fft-libero'}
HEADS = ('progress_head.', 'preference_head.', 'success_head.', 'similarity_head.', 'frame_pool_attn.')

p = argparse.ArgumentParser()
p.add_argument('--judge', choices=list(JUDGES), required=True)
p.add_argument('--traj', required=True, help='glob of trajectory .hdf5 files')
p.add_argument('--out', required=True)
p.add_argument('--robometer_repo', default='/content/robometer')
p.add_argument('--frames', type=int, default=8)
a = p.parse_args()

sys.path.insert(0, a.robometer_repo)
sys.modules.setdefault('unsloth', types.SimpleNamespace(FastVisionModel=None))
from huggingface_hub import snapshot_download
from robometer.configs.experiment_configs import ExperimentConfig
from robometer.data.dataset_types import ProgressSample, Trajectory
from robometer.evals.eval_server import compute_batch_outputs
from robometer.models import RBM
from robometer.utils.setup_utils import _add_special_tokens_and_resize, _setup_processor_and_tokenizer, setup_batch_collator
from safetensors import safe_open
from transformers import Qwen3VLModel


def load_judge(model_id):
    ckpt = Path(snapshot_download(model_id, allow_patterns=['*.json', '*.yaml', '*.safetensors']))

    class Loader(yaml.SafeLoader):
        pass

    Loader.add_constructor('tag:yaml.org,2002:python/object:robometer.models.rewind_transformer.ReWINDTransformerConfig',
                           lambda loader, node: loader.construct_mapping(node))
    raw = yaml.load((ckpt / 'config.yaml').read_text(), Loader=Loader)
    cfg = ExperimentConfig(**{k: v for k, v in raw.items() if k in {f.name for f in fields(ExperimentConfig)}})
    cfg.model.use_unsloth = False
    cfg.data.use_multi_image = True
    processor = _setup_processor_and_tokenizer(cfg.model)
    base = Qwen3VLModel.from_pretrained(ckpt, dtype=torch.bfloat16, attn_implementation='sdpa', device_map={'': 'cuda'}, key_mapping={r'^model\.': ''})
    _add_special_tokens_and_resize(cfg.model, processor, base)
    model = RBM(config=base.config, processor=processor, tokenizer=processor.tokenizer, base_model=base,
                base_model_id=cfg.model.base_model_id, model_config=cfg.model)
    heads = {}
    for shard in sorted(ckpt.glob('model-*.safetensors')):
        with safe_open(str(shard), framework='pt') as f:
            heads.update({k: f.get_tensor(k) for k in f.keys() if k.startswith(HEADS)})
    own = model.state_dict().keys()
    model.load_state_dict({k: v for k, v in heads.items() if k in own}, strict=False)
    for name in ('progress_head', 'preference_head', 'success_head', 'frame_pool_attn'):
        if hasattr(model, name):
            getattr(model, name).to(device='cuda', dtype=torch.bfloat16)
    collator = setup_batch_collator(processor, processor.tokenizer, cfg, is_eval=True)
    discrete = getattr(cfg.loss, 'progress_loss_type', 'l2').lower() == 'discrete'
    return model.eval(), processor.tokenizer, collator, discrete, getattr(cfg.loss, 'progress_discrete_bins', None) or 10


model, tokenizer, collator, discrete, bins = load_judge(JUDGES[a.judge])


@torch.no_grad()
def score(frames, task):
    traj = Trajectory(frames=frames, frames_shape=tuple(frames.shape), task=task, id='0', metadata={'subsequence_length': len(frames)})
    inputs = collator([ProgressSample(trajectory=traj, sample_type='progress')])['progress_inputs']
    inputs = {k: v.to('cuda') if hasattr(v, 'to') else v for k, v in inputs.items()}
    out = compute_batch_outputs(model, tokenizer, inputs, sample_type='progress', is_discrete_mode=discrete, num_bins=bins)
    return np.clip(np.asarray(out['progress_pred'][0], float), 0, 1).tolist(), np.asarray(out['outputs_success']['success_probs'][0], float).tolist()


done = set()
if os.path.exists(a.out):
    done = {(r['task'], r['ep'], r['rseed']) for r in map(json.loads, open(a.out))}
files = sorted(glob.glob(a.traj))
print(len(files), 'trajectories,', len(done), 'already scored', flush=True)
for n, path in enumerate(files):
    m = re.search(r'task(\d+)_ep(\d+)_r(\d+)_success(\d)', os.path.basename(path))
    key = tuple(int(m.group(i)) for i in (1, 2, 3))
    if key in done:
        continue
    with h5py.File(path, 'r') as h:
        task = h.attrs['task_description']
        task = task.decode() if isinstance(task, bytes) else str(task)
        idx = np.unique(np.linspace(0, len(h['primary_images_jpeg']) - 1, a.frames).round().astype(int))
        frames = np.stack([np.asarray(Image.open(io.BytesIO(h['primary_images_jpeg'][i].tobytes())).convert('RGB').resize((224, 224), Image.BILINEAR)) for i in idx])
    prog, succ = score(frames, task)
    row = dict(task=key[0], ep=key[1], rseed=key[2], success=bool(int(m.group(4))), progress=prog, success_prob=succ)
    with open(a.out, 'a') as f:
        f.write(json.dumps(row) + '\n')
    if n % 100 == 0:
        print(n, flush=True)
print('SCORED', len(files), flush=True)

Best attempt per start state, by the judge's final progress, kept if at least 0.5.

In [ ]:
%%writefile /content/code/select_best.py
import argparse
import glob
import json
import os
import shutil

import h5py
import numpy as np

p = argparse.ArgumentParser()
p.add_argument('--scores', required=True)
p.add_argument('--traj_dirs', required=True, help='glob of the run traj/ folders the scores came from')
p.add_argument('--out', required=True)
p.add_argument('--by', choices=['progress', 'success_prob', 'true_success'], default='progress')
p.add_argument('--threshold', type=float, default=0.5)
p.add_argument('--suite', default='libero_spatial')
a = p.parse_args()

paths = {}
for d in glob.glob(a.traj_dirs):
    for f in glob.glob(os.path.join(d, '*.hdf5')):
        name = os.path.basename(f)
        t, e, r = name.split('_')[:3]
        paths[(int(t[4:]), int(e[2:]), int(r[1:]))] = f

groups = {}
for r in map(json.loads, open(a.scores)):
    groups.setdefault((r['task'], r['ep']), []).append(r)
score = (lambda r: float(r['success'])) if a.by == 'true_success' else (lambda r: r[a.by][-1])

out = os.path.join(a.out, f'suite={a.suite}')
shutil.rmtree(a.out, ignore_errors=True)
os.makedirs(out)
kept = []
for state, rows in sorted(groups.items()):
    best = max(rows, key=score)
    if score(best) < a.threshold:
        continue
    dst = os.path.join(out, os.path.basename(paths[(best['task'], best['ep'], best['rseed'])]))
    shutil.copyfile(paths[(best['task'], best['ep'], best['rseed'])], dst)
    with h5py.File(dst, 'a') as f:
        f.attrs['success'] = True
    kept.append(best)
print(f'kept {len(kept)} of {len(groups)} start states; truly successful among them {np.mean([r["success"] for r in kept]):.3f}')

Success per run, and per judge: AUROC, best-of-N pick success and the oracle ceiling.

In [ ]:
%%writefile /content/code/report.py
import argparse
import json
import os

import numpy as np

p = argparse.ArgumentParser()
p.add_argument('--runs', nargs='*', default=[], help='rollout output folders; the first one is the baseline for paired differences')
p.add_argument('--scores', nargs='*', default=[], help='judge score files')
a = p.parse_args()


def auroc(s, y):
    s, y = np.asarray(s), np.asarray(y, bool)
    pos, neg = s[y], s[~y]
    return float(((pos[:, None] > neg[None]) + 0.5 * (pos[:, None] == neg[None])).mean()) if len(pos) and len(neg) else float('nan')


def boot_ci(x, n=5000):
    rng = np.random.default_rng(0)
    m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


if a.runs:
    runs = {r: {(e['task'], e['ep']): e['success'] for e in map(json.loads, open(os.path.join(r, 'episodes.jsonl')))} for r in a.runs}
    base = runs[a.runs[0]]
    print('| run | episodes | success | per task | vs first run (paired, 95% CI) |\n|---|---|---|---|---|')
    for r, d in runs.items():
        tasks = sorted({t for t, _ in d})
        per = ' '.join(f'{np.mean([v for (t, _), v in d.items() if t == k]):.2f}' for k in tasks)
        shared = [k for k in d if k in base]
        diff = np.array([d[k] - base[k] for k in shared], float)
        lo, hi = boot_ci(diff) if r != a.runs[0] and len(diff) else (0, 0)
        cmp = '-' if r == a.runs[0] else f'{diff.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]'
        print(f'| {os.path.basename(r.rstrip("/"))} | {len(d)} | {np.mean(list(d.values())):.3f} | {per} | {cmp} |')

for f in a.scores:
    rows = [json.loads(l) for l in open(f)]
    groups = {}
    for r in rows:
        groups.setdefault((r['task'], r['ep']), []).append(r)
    k = max(len(g) for g in groups.values())
    groups = {s: g for s, g in groups.items() if len(g) == k}
    single = np.mean([r['success'] for g in groups.values() for r in g])
    oracle = np.mean([any(r['success'] for r in g) for g in groups.values()])
    mixed = [g for g in groups.values() if 0 < sum(r['success'] for r in g) < len(g)]
    print(f'\n{os.path.basename(f)}: {len(groups)} start states x {k} attempts; single attempt {single:.3f}, best-of-{k} ceiling {oracle:.3f}, mixed states {len(mixed)}')
    print(f'| score | AUROC | best-of-{k} pick success | picks a success on mixed states |\n|---|---|---|---|')
    for key in ('progress', 'success_prob'):
        pick = lambda g: max(g, key=lambda r: r[key][-1])['success']
        print(f'| final {key} | {auroc([r[key][-1] for r in rows], [r["success"] for r in rows]):.3f} | '
              f'{np.mean([pick(g) for g in groups.values()]):.3f} | {sum(pick(g) for g in mixed)}/{len(mixed)} |')

## Setup

In [ ]:
!bash $CODE/setup_colab.sh 1

## Build the custom tasks (skipped for a LIBERO suite)

In [ ]:
%%bash
if [ -z "${TASK_ARGS##--tasks_manifest*}" ]; then
  source /content/env.sh
  $CPV $CODE/tasks/make_tasks.py 50 2>&1 | grep valid
  mkdir -p $OUT/tasks && cp -r $CODE/tasks/* $OUT/tasks/
fi

## Rollouts on training start states 0-24 (with trajectories)

In [ ]:
%%bash
source /content/env.sh
cd /content/cosmos-policy
mkdir -p /content/runs
for r in $(seq 0 $((SEEDS - 1))); do
  $CPV $CODE/rollout.py --out /content/runs/collect_r$r $TASK_ARGS --episodes 0:25 --rseed $r --save_traj 1 > /content/runs/collect_r$r.log 2>&1
  tail -n 1 /content/runs/collect_r$r.log
  mkdir -p $OUT/collect_r$r && cp /content/runs/collect_r$r/episodes.jsonl $OUT/collect_r$r/
done

## Base policy on held-out start states 25-49 (the baseline for the fine-tune)

In [ ]:
%%bash
source /content/env.sh
cd /content/cosmos-policy
for r in $(seq 0 $((HELDOUT_SEEDS - 1))); do
  $CPV $CODE/rollout.py --out /content/runs/base_heldout_r$r $TASK_ARGS --episodes 25:50 --rseed $r > /content/runs/base_heldout_r$r.log 2>&1
  tail -n 1 /content/runs/base_heldout_r$r.log
  mkdir -p $OUT/base_heldout_r$r && cp /content/runs/base_heldout_r$r/episodes.jsonl $OUT/base_heldout_r$r/
done

## Score every rollout with both judges

In [ ]:
%%bash
source /content/env.sh
cd /content/robometer
for j in base libero; do
  $RBV $CODE/judge.py --judge $j --traj "/content/runs/collect_r*/traj/*.hdf5" --out $OUT/scores_$j.jsonl --robometer_repo /content/robometer 2>&1 | tail -n 1
done

In [ ]:
%%bash
source /content/env.sh
$CPV $CODE/report.py --runs $(ls -d $OUT/collect_r*) --scores $OUT/scores_base.jsonl $OUT/scores_libero.jsonl

## Build the fine-tune set

The judge's best attempt per start state, kept if its final progress is at least 0.5.

In [ ]:
%%bash
source /content/env.sh
$CPV $CODE/select_best.py --scores $OUT/scores_$SELECT_WITH.jsonl --traj_dirs "/content/runs/collect_r*/traj" --out /content/ftset
tar cf $OUT/ftset_$SELECT_WITH.tar -C /content ftset
ls -la $OUT/ftset_$SELECT_WITH.tar